### FAISS ( Facebook AI Similarity Search)

In [1]:
from langchain_community.document_loaders import TextLoader
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size = 600, chunk_overlap = 0)

loader1 = TextLoader("nlp-keywords.txt",encoding="utf-8")
loader2 = TextLoader("finance-keywords.txt", encoding="utf-8")

split_doc1 = loader1.load_and_split(text_splitter)
split_doc2 = loader2.load_and_split(text_splitter)


len(split_doc1), len(split_doc2)

(11, 6)

In [2]:
import faiss
from langchain_community.vectorstores import FAISS
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_openai import OpenAIEmbeddings


embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 임베딩 차원 크기를 계산
dimension_size = len(embeddings.embed_query("hello world"))
print(dimension_size)

1536


In [3]:
db = FAISS(
    embedding_function= embeddings,
    index=faiss.IndexFlatL2(dimension_size),
    docstore= InMemoryDocstore(),
    index_to_docstore_id={}
)

In [4]:
db = FAISS.from_documents(documents=split_doc1, embedding=OpenAIEmbeddings())

In [5]:
db.index_to_docstore_id

{0: '8d2692a8-784b-4aef-9823-d92853887cb3',
 1: 'b875d24c-b93c-4f29-94d9-e68336a69bf7',
 2: '04f20754-d628-42f6-8386-61d7824f65b4',
 3: 'e2ead1c3-b0ef-4ef3-b279-8270d4caa786',
 4: 'dba4d01d-af91-4464-94f5-056b04efe73c',
 5: '5ed1fb39-f0fe-41f8-8660-d66971ad4333',
 6: '0aea28ef-dd10-4c52-bf23-b111204f9c86',
 7: '68d62e80-6aed-4d3b-b0ca-5a2cb43252fa',
 8: '254bae3b-7d2a-4a83-ad36-0a632da0bd5e',
 9: '4f932883-5af6-4dfe-89d1-7be4669231e2',
 10: '6c495cb8-9fa1-4285-b10f-32953b5f9a1a'}

In [ ]:
db.docstore._dict   # 딕셔너리 형태로 제공

{'8d2692a8-784b-4aef-9823-d92853887cb3': Document(id='8d2692a8-784b-4aef-9823-d92853887cb3', metadata={'source': 'nlp-keywords.txt'}, page_content='Semantic Search\n\n정의: 의미론적 검색은 사용자의 질의를 단순한 키워드 매칭을 넘어서 그 의미를 파악하여 관련된 결과를 반환하는 검색 방식입니다.\n예시: 사용자가 "태양계 행성"이라고 검색하면, "목성", "화성" 등과 같이 관련된 행성에 대한 정보를 반환합니다.\n연관키워드: 자연어 처리, 검색 알고리즘, 데이터 마이닝\n\nEmbedding\n\n정의: 임베딩은 단어나 문장 같은 텍스트 데이터를 저차원의 연속적인 벡터로 변환하는 과정입니다. 이를 통해 컴퓨터가 텍스트를 이해하고 처리할 수 있게 합니다.\n예시: "사과"라는 단어를 [0.65, -0.23, 0.17]과 같은 벡터로 표현합니다.\n연관키워드: 자연어 처리, 벡터화, 딥러닝\n\nToken\n\n정의: 토큰은 텍스트를 더 작은 단위로 분할하는 것을 의미합니다. 이는 일반적으로 단어, 문장, 또는 구절일 수 있습니다.\n예시: 문장 "나는 학교에 간다"를 "나는", "학교에", "간다"로 분할합니다.\n연관키워드: 토큰화, 자연어 처리, 구문 분석\n\nTokenizer'),
 'b875d24c-b93c-4f29-94d9-e68336a69bf7': Document(id='b875d24c-b93c-4f29-94d9-e68336a69bf7', metadata={'source': 'nlp-keywords.txt'}, page_content='정의: 토크나이저는 텍스트 데이터를 토큰으로 분할하는 도구입니다. 이는 자연어 처리에서 데이터를 전처리하는 데 사용됩니다.\n예시: "I love programming."이라는 문장을 ["I", "love", "programming", "."]으로 분할합니다.\n연관키워드: 토큰화, 자연

In [7]:
db2 = FAISS.from_texts(
    ["안녕하세요. 정말 반갑습니다.", "제 이름은 테디입니다."],
    embedding= OpenAIEmbeddings(),
    metadatas=[{"source":"텍스트문서"}, {"source":"텍스트문서"}],
    ids = ["doc1", "doc2"]
)

In [8]:
db2.docstore._dict

{'doc1': Document(id='doc1', metadata={'source': '텍스트문서'}, page_content='안녕하세요. 정말 반갑습니다.'),
 'doc2': Document(id='doc2', metadata={'source': '텍스트문서'}, page_content='제 이름은 테디입니다.')}

In [9]:
db.similarity_search("TF IDF에 대하여 알려줘") #유사도 검색

[Document(id='0aea28ef-dd10-4c52-bf23-b111204f9c86', metadata={'source': 'nlp-keywords.txt'}, page_content='정의: TF-IDF는 문서 내에서 단어의 중요도를 평가하는 데 사용되는 통계적 척도입니다. 이는 문서 내 단어의 빈도와 전체 문서 집합에서 그 단어의 희소성을 고려합니다.\n예시: 많은 문서에서 자주 등장하지 않는 단어는 높은 TF-IDF 값을 가집니다.\n연관키워드: 자연어 처리, 정보 검색, 데이터 마이닝\n\nDeep Learning\n\n정의: 딥러닝은 인공신경망을 이용하여 복잡한 문제를 해결하는 머신러닝의 한 분야입니다. 이는 데이터에서 고수준의 표현을 학습하는 데 중점을 둡니다.\n예시: 이미지 인식, 음성 인식, 자연어 처리 등에서 딥러닝 모델이 활용됩니다.\n연관키워드: 인공신경망, 머신러닝, 데이터 분석\n\nSchema\n\n정의: 스키마는 데이터베이스나 파일의 구조를 정의하는 것으로, 데이터가 어떻게 저장되고 조직되는지에 대한 청사진을 제공합니다.\n예시: 관계형 데이터베이스의 테이블 스키마는 열 이름, 데이터 타입, 키 제약 조건 등을 정의합니다.\n연관키워드: 데이터베이스, 데이터 모델링, 데이터 관리\n\nDataFrame'),
 Document(id='5ed1fb39-f0fe-41f8-8660-d66971ad4333', metadata={'source': 'nlp-keywords.txt'}, page_content='정의: 오픈 소스는 소스 코드가 공개되어 누구나 자유롭게 사용, 수정, 배포할 수 있는 소프트웨어를 의미합니다. 이는 협업과 혁신을 촉진하는 데 중요한 역할을 합니다.\n예시: 리눅스 운영 체제는 대표적인 오픈 소스 프로젝트입니다.\n연관키워드: 소프트웨어 개발, 커뮤니티, 기술 협업\n\nStructured Data\n\n정의: 구조화된 데이터는 정해진 형식이나 스키마에 따라 조직된 데이터입니다. 이는 데이터베이스, 스프레드시트 

In [10]:
db.similarity_search("TF IDF에 대하여 알려줘", k=2)

[Document(id='0aea28ef-dd10-4c52-bf23-b111204f9c86', metadata={'source': 'nlp-keywords.txt'}, page_content='정의: TF-IDF는 문서 내에서 단어의 중요도를 평가하는 데 사용되는 통계적 척도입니다. 이는 문서 내 단어의 빈도와 전체 문서 집합에서 그 단어의 희소성을 고려합니다.\n예시: 많은 문서에서 자주 등장하지 않는 단어는 높은 TF-IDF 값을 가집니다.\n연관키워드: 자연어 처리, 정보 검색, 데이터 마이닝\n\nDeep Learning\n\n정의: 딥러닝은 인공신경망을 이용하여 복잡한 문제를 해결하는 머신러닝의 한 분야입니다. 이는 데이터에서 고수준의 표현을 학습하는 데 중점을 둡니다.\n예시: 이미지 인식, 음성 인식, 자연어 처리 등에서 딥러닝 모델이 활용됩니다.\n연관키워드: 인공신경망, 머신러닝, 데이터 분석\n\nSchema\n\n정의: 스키마는 데이터베이스나 파일의 구조를 정의하는 것으로, 데이터가 어떻게 저장되고 조직되는지에 대한 청사진을 제공합니다.\n예시: 관계형 데이터베이스의 테이블 스키마는 열 이름, 데이터 타입, 키 제약 조건 등을 정의합니다.\n연관키워드: 데이터베이스, 데이터 모델링, 데이터 관리\n\nDataFrame'),
 Document(id='5ed1fb39-f0fe-41f8-8660-d66971ad4333', metadata={'source': 'nlp-keywords.txt'}, page_content='정의: 오픈 소스는 소스 코드가 공개되어 누구나 자유롭게 사용, 수정, 배포할 수 있는 소프트웨어를 의미합니다. 이는 협업과 혁신을 촉진하는 데 중요한 역할을 합니다.\n예시: 리눅스 운영 체제는 대표적인 오픈 소스 프로젝트입니다.\n연관키워드: 소프트웨어 개발, 커뮤니티, 기술 협업\n\nStructured Data\n\n정의: 구조화된 데이터는 정해진 형식이나 스키마에 따라 조직된 데이터입니다. 이는 데이터베이스, 스프레드시트 

In [11]:
# filter 사용
db.similarity_search(
    "TF IDF에 대하여 알려줘", filter={"source":"nlp-keywords.txt"}, k=2
)

[Document(id='0aea28ef-dd10-4c52-bf23-b111204f9c86', metadata={'source': 'nlp-keywords.txt'}, page_content='정의: TF-IDF는 문서 내에서 단어의 중요도를 평가하는 데 사용되는 통계적 척도입니다. 이는 문서 내 단어의 빈도와 전체 문서 집합에서 그 단어의 희소성을 고려합니다.\n예시: 많은 문서에서 자주 등장하지 않는 단어는 높은 TF-IDF 값을 가집니다.\n연관키워드: 자연어 처리, 정보 검색, 데이터 마이닝\n\nDeep Learning\n\n정의: 딥러닝은 인공신경망을 이용하여 복잡한 문제를 해결하는 머신러닝의 한 분야입니다. 이는 데이터에서 고수준의 표현을 학습하는 데 중점을 둡니다.\n예시: 이미지 인식, 음성 인식, 자연어 처리 등에서 딥러닝 모델이 활용됩니다.\n연관키워드: 인공신경망, 머신러닝, 데이터 분석\n\nSchema\n\n정의: 스키마는 데이터베이스나 파일의 구조를 정의하는 것으로, 데이터가 어떻게 저장되고 조직되는지에 대한 청사진을 제공합니다.\n예시: 관계형 데이터베이스의 테이블 스키마는 열 이름, 데이터 타입, 키 제약 조건 등을 정의합니다.\n연관키워드: 데이터베이스, 데이터 모델링, 데이터 관리\n\nDataFrame'),
 Document(id='5ed1fb39-f0fe-41f8-8660-d66971ad4333', metadata={'source': 'nlp-keywords.txt'}, page_content='정의: 오픈 소스는 소스 코드가 공개되어 누구나 자유롭게 사용, 수정, 배포할 수 있는 소프트웨어를 의미합니다. 이는 협업과 혁신을 촉진하는 데 중요한 역할을 합니다.\n예시: 리눅스 운영 체제는 대표적인 오픈 소스 프로젝트입니다.\n연관키워드: 소프트웨어 개발, 커뮤니티, 기술 협업\n\nStructured Data\n\n정의: 구조화된 데이터는 정해진 형식이나 스키마에 따라 조직된 데이터입니다. 이는 데이터베이스, 스프레드시트 

In [12]:
db.similarity_search(
    "TF IDF 에 대하여 알려줘", filter={"source": "finance-keywords.txt"}, k=2
)

[]

In [13]:
from langchain_core.documents import Document

db.add_documents(
    [
        Document(
            page_content="안녕하세요! 이번엔 도큐먼트를 새로 추가해 볼게요",
            metadata= {"source": "mydata.txt"}
        )
    ],
    ids= ["new_doc1"]
)

['new_doc1']

In [14]:
db.similarity_search("안녕하세요", k=1)

[Document(id='new_doc1', metadata={'source': 'mydata.txt'}, page_content='안녕하세요! 이번엔 도큐먼트를 새로 추가해 볼게요')]

In [15]:
db.add_texts(
    ["이번엔 텍스트 데이터를 추가합니다.", "추가한 2번째 텍스트 데이터 입니다."],
    metadatas=[{"source": "mydata.txt"},{"source":"mydata.txt"}],
    ids=["new_doc2", "new_doc3"]
)

['new_doc2', 'new_doc3']

In [16]:
db.index_to_docstore_id

{0: '8d2692a8-784b-4aef-9823-d92853887cb3',
 1: 'b875d24c-b93c-4f29-94d9-e68336a69bf7',
 2: '04f20754-d628-42f6-8386-61d7824f65b4',
 3: 'e2ead1c3-b0ef-4ef3-b279-8270d4caa786',
 4: 'dba4d01d-af91-4464-94f5-056b04efe73c',
 5: '5ed1fb39-f0fe-41f8-8660-d66971ad4333',
 6: '0aea28ef-dd10-4c52-bf23-b111204f9c86',
 7: '68d62e80-6aed-4d3b-b0ca-5a2cb43252fa',
 8: '254bae3b-7d2a-4a83-ad36-0a632da0bd5e',
 9: '4f932883-5af6-4dfe-89d1-7be4669231e2',
 10: '6c495cb8-9fa1-4285-b10f-32953b5f9a1a',
 11: 'new_doc1',
 12: 'new_doc2',
 13: 'new_doc3'}

In [17]:
ids = db.add_texts(
    ["삭제용 데이터를 추가합니다.", "2번째 삭제용 데이터입니다."],
    metadatas=[{"source":"mydata.txt"}, {"source":"mydata.txt"}],
    ids=["delete_doc1", "delete_doc2"]
)

print(ids)

['delete_doc1', 'delete_doc2']


In [18]:
db.delete(ids)
db.index_to_docstore_id

{0: '8d2692a8-784b-4aef-9823-d92853887cb3',
 1: 'b875d24c-b93c-4f29-94d9-e68336a69bf7',
 2: '04f20754-d628-42f6-8386-61d7824f65b4',
 3: 'e2ead1c3-b0ef-4ef3-b279-8270d4caa786',
 4: 'dba4d01d-af91-4464-94f5-056b04efe73c',
 5: '5ed1fb39-f0fe-41f8-8660-d66971ad4333',
 6: '0aea28ef-dd10-4c52-bf23-b111204f9c86',
 7: '68d62e80-6aed-4d3b-b0ca-5a2cb43252fa',
 8: '254bae3b-7d2a-4a83-ad36-0a632da0bd5e',
 9: '4f932883-5af6-4dfe-89d1-7be4669231e2',
 10: '6c495cb8-9fa1-4285-b10f-32953b5f9a1a',
 11: 'new_doc1',
 12: 'new_doc2',
 13: 'new_doc3'}

In [19]:
db.save_local(folder_path="faiss_db", index_name="faiss_index")

In [20]:
loaded_db = FAISS.load_local(
    folder_path="faiss_db",
    index_name="faiss_index",
    embeddings= embeddings,
    allow_dangerous_deserialization= True
)

loaded_db.index_to_docstore_id

{0: '8d2692a8-784b-4aef-9823-d92853887cb3',
 1: 'b875d24c-b93c-4f29-94d9-e68336a69bf7',
 2: '04f20754-d628-42f6-8386-61d7824f65b4',
 3: 'e2ead1c3-b0ef-4ef3-b279-8270d4caa786',
 4: 'dba4d01d-af91-4464-94f5-056b04efe73c',
 5: '5ed1fb39-f0fe-41f8-8660-d66971ad4333',
 6: '0aea28ef-dd10-4c52-bf23-b111204f9c86',
 7: '68d62e80-6aed-4d3b-b0ca-5a2cb43252fa',
 8: '254bae3b-7d2a-4a83-ad36-0a632da0bd5e',
 9: '4f932883-5af6-4dfe-89d1-7be4669231e2',
 10: '6c495cb8-9fa1-4285-b10f-32953b5f9a1a',
 11: 'new_doc1',
 12: 'new_doc2',
 13: 'new_doc3'}

### FAISS 객체 병합하기

In [21]:
db = FAISS.load_local(
    folder_path="faiss_db",
    index_name="faiss_index",
    embeddings= embeddings,
    allow_dangerous_deserialization= True
)

In [22]:
db2 = FAISS.from_documents(documents= split_doc2, embedding= OpenAIEmbeddings())

In [23]:
db.index_to_docstore_id

{0: '8d2692a8-784b-4aef-9823-d92853887cb3',
 1: 'b875d24c-b93c-4f29-94d9-e68336a69bf7',
 2: '04f20754-d628-42f6-8386-61d7824f65b4',
 3: 'e2ead1c3-b0ef-4ef3-b279-8270d4caa786',
 4: 'dba4d01d-af91-4464-94f5-056b04efe73c',
 5: '5ed1fb39-f0fe-41f8-8660-d66971ad4333',
 6: '0aea28ef-dd10-4c52-bf23-b111204f9c86',
 7: '68d62e80-6aed-4d3b-b0ca-5a2cb43252fa',
 8: '254bae3b-7d2a-4a83-ad36-0a632da0bd5e',
 9: '4f932883-5af6-4dfe-89d1-7be4669231e2',
 10: '6c495cb8-9fa1-4285-b10f-32953b5f9a1a',
 11: 'new_doc1',
 12: 'new_doc2',
 13: 'new_doc3'}

In [24]:
db2.index_to_docstore_id

{0: '8f7327ec-10e0-4800-886f-30c8901a1aad',
 1: 'c7df34db-0e4d-4d91-855f-c6781ddd8592',
 2: '46aabbbf-848a-43f1-8668-0c94b68d7c2e',
 3: 'bc3b35fb-caeb-49df-b821-e2036628a30e',
 4: '67a25c0b-881f-4115-ad12-ffbc2cec996f',
 5: '8afe01ae-0f64-437c-a0f3-9b7c65a50d0e'}

In [25]:
db.merge_from(db2)
db.index_to_docstore_id

{0: '8d2692a8-784b-4aef-9823-d92853887cb3',
 1: 'b875d24c-b93c-4f29-94d9-e68336a69bf7',
 2: '04f20754-d628-42f6-8386-61d7824f65b4',
 3: 'e2ead1c3-b0ef-4ef3-b279-8270d4caa786',
 4: 'dba4d01d-af91-4464-94f5-056b04efe73c',
 5: '5ed1fb39-f0fe-41f8-8660-d66971ad4333',
 6: '0aea28ef-dd10-4c52-bf23-b111204f9c86',
 7: '68d62e80-6aed-4d3b-b0ca-5a2cb43252fa',
 8: '254bae3b-7d2a-4a83-ad36-0a632da0bd5e',
 9: '4f932883-5af6-4dfe-89d1-7be4669231e2',
 10: '6c495cb8-9fa1-4285-b10f-32953b5f9a1a',
 11: 'new_doc1',
 12: 'new_doc2',
 13: 'new_doc3',
 14: '8f7327ec-10e0-4800-886f-30c8901a1aad',
 15: 'c7df34db-0e4d-4d91-855f-c6781ddd8592',
 16: '46aabbbf-848a-43f1-8668-0c94b68d7c2e',
 17: 'bc3b35fb-caeb-49df-b821-e2036628a30e',
 18: '67a25c0b-881f-4115-ad12-ffbc2cec996f',
 19: '8afe01ae-0f64-437c-a0f3-9b7c65a50d0e'}

### 리트리버로 변환하기

In [26]:
db = FAISS.from_documents(
    documents= split_doc1 + split_doc2,
    embedding= OpenAIEmbeddings()
)

In [27]:
retriever = db.as_retriever()
retriever.invoke("Word2Vec에 대하여 알려줘")

[Document(id='a87ff8ef-25ea-4c8c-ae20-173bd2c8a10a', metadata={'source': 'nlp-keywords.txt'}, page_content='정의: Word2Vec은 단어를 벡터 공간에 매핑하여 단어 간의 의미적 관계를 나타내는 자연어 처리 기술입니다. 이는 단어의 문맥적 유사성을 기반으로 벡터를 생성합니다.\n예시: Word2Vec 모델에서 "왕"과 "여왕"은 서로 가까운 위치에 벡터로 표현됩니다.\n연관키워드: 자연어 처리, 임베딩, 의미론적 유사성\nLLM (Large Language Model)\n\n정의: LLM은 대규모의 텍스트 데이터로 훈련된 큰 규모의 언어 모델을 의미합니다. 이러한 모델은 다양한 자연어 이해 및 생성 작업에 사용됩니다.\n예시: OpenAI의 GPT 시리즈는 대표적인 대규모 언어 모델입니다.\n연관키워드: 자연어 처리, 딥러닝, 텍스트 생성\n\nFAISS (Facebook AI Similarity Search)\n\n정의: FAISS는 페이스북에서 개발한 고속 유사성 검색 라이브러리로, 특히 대규모 벡터 집합에서 유사 벡터를 효과적으로 검색할 수 있도록 설계되었습니다.\n예시: 수백만 개의 이미지 벡터 중에서 비슷한 이미지를 빠르게 찾는 데 FAISS가 사용될 수 있습니다.\n연관키워드: 벡터 검색, 머신러닝, 데이터베이스 최적화\n\nOpen Source'),
 Document(id='a83bfbfa-7190-465e-935c-7f2310c05108', metadata={'source': 'nlp-keywords.txt'}, page_content='정의: HuggingFace는 자연어 처리를 위한 다양한 사전 훈련된 모델과 도구를 제공하는 라이브러리입니다. 이는 연구자와 개발자들이 쉽게 NLP 작업을 수행할 수 있도록 돕습니다.\n예시: HuggingFace의 Transformers 라이브러리를 사용하여 감정 분석, 텍스트 생성 등의 작업을 수행할 수 있습니다.\n연관키워

In [28]:
retriever = db.as_retriever(
    search_type = "mmr",
    search_kwargs = {"k":6 , "lambda_mult":0.25, "fetch_k":10}
)

retriever.invoke("Word2Vec에 대하여 알려줘")

[Document(id='a87ff8ef-25ea-4c8c-ae20-173bd2c8a10a', metadata={'source': 'nlp-keywords.txt'}, page_content='정의: Word2Vec은 단어를 벡터 공간에 매핑하여 단어 간의 의미적 관계를 나타내는 자연어 처리 기술입니다. 이는 단어의 문맥적 유사성을 기반으로 벡터를 생성합니다.\n예시: Word2Vec 모델에서 "왕"과 "여왕"은 서로 가까운 위치에 벡터로 표현됩니다.\n연관키워드: 자연어 처리, 임베딩, 의미론적 유사성\nLLM (Large Language Model)\n\n정의: LLM은 대규모의 텍스트 데이터로 훈련된 큰 규모의 언어 모델을 의미합니다. 이러한 모델은 다양한 자연어 이해 및 생성 작업에 사용됩니다.\n예시: OpenAI의 GPT 시리즈는 대표적인 대규모 언어 모델입니다.\n연관키워드: 자연어 처리, 딥러닝, 텍스트 생성\n\nFAISS (Facebook AI Similarity Search)\n\n정의: FAISS는 페이스북에서 개발한 고속 유사성 검색 라이브러리로, 특히 대규모 벡터 집합에서 유사 벡터를 효과적으로 검색할 수 있도록 설계되었습니다.\n예시: 수백만 개의 이미지 벡터 중에서 비슷한 이미지를 빠르게 찾는 데 FAISS가 사용될 수 있습니다.\n연관키워드: 벡터 검색, 머신러닝, 데이터베이스 최적화\n\nOpen Source'),
 Document(id='a30ec578-f903-4d09-8346-4e315086e508', metadata={'source': 'nlp-keywords.txt'}, page_content='GPT (Generative Pretrained Transformer)\n\n정의: GPT는 대규모의 데이터셋으로 사전 훈련된 생성적 언어 모델로, 다양한 텍스트 기반 작업에 활용됩니다. 이는 입력된 텍스트에 기반하여 자연스러운 언어를 생성할 수 있습니다.\n예시: 사용자가 제공한 질문에 대해 자세한 답변을 생성하는 챗봇은 GP

In [29]:
retriever = db.as_retriever(
    search_type= "similarity_score_threshold",
    search_kwargs= {"source_threshold":0.8}
)

retriever.invoke("Word2Vec 에 대하여 알려줘")

ValidationError: 1 validation error for VectorStoreRetriever
  Value error, `score_threshold` is not specified with a float value(0~1) in `search_kwargs`. [type=value_error, input_value={'vectorstore': <langchai...source_threshold': 0.8}}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error